In [ ]:
import json
import sys
from pathlib import Path

HICOLA_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'hicola_paths.py').exists())
sys.path.insert(0, str(HICOLA_ROOT / 'scripts'))
import hicola_paths as hp
from halo_pipeline import require_peculiar_velocities  # halo catalogs made before the velocity fix are refused
sys.path.insert(0, str(hp.HODDIES_ROOT))
for module_name in list(sys.modules):
    if module_name == 'HODDIES' or module_name.startswith('HODDIES.'):
        del sys.modules[module_name]

import numpy as np
import matplotlib.pyplot as plt
from HODDIES import HOD
from HODDIES.sim_loader import Base_catalogue
from euclid_reference_cosmology import SERIALIZABLE_PARAMETERS, PARAMETERS
import HODDIES

print(f'Using HODDIES from {HODDIES.__file__}')


In [ ]:
from cosmoprimo.fiducial import Cosmology

euclid_cosmo = Cosmology(
    Omega0_m=SERIALIZABLE_PARAMETERS['Omega_m'],
    Omega_b=SERIALIZABLE_PARAMETERS['Omega_b'],
    h=SERIALIZABLE_PARAMETERS['h'],
    sigma8=SERIALIZABLE_PARAMETERS['sigma8'],
    n_s=SERIALIZABLE_PARAMETERS['n_s'],
    w0_fld=SERIALIZABLE_PARAMETERS['w0'],
    wa_fld=SERIALIZABLE_PARAMETERS['wa'],
    T_cmb=SERIALIZABLE_PARAMETERS['Tcmb0_K'],
    N_eff=SERIALIZABLE_PARAMETERS['Neff'],
    m_ncdm=[],
    engine='class',
)

print(SERIALIZABLE_PARAMETERS)
euclid_cosmo.get_params()


In [ ]:
target, model, realization = 'COLA_NM4096', 'LCDM', 's1234'
redshift = 1.650

OUTPUT_DIR = hp.halo_catalog_dir(target, model, realization, redshift)
HALO_CATALOG = OUTPUT_DIR / 'halos200c_catalog.npy'
HALO_METADATA = Path(str(HALO_CATALOG) + '.json')
HODDIES_CATALOG = OUTPUT_DIR / 'halos200c_hoddies.npy'
HMF_OUTPUT = OUTPUT_DIR / 'halos200c_hmf.txt'
ANALYSIS_DIR = OUTPUT_DIR / 'analysis'
PARAM_FILE = str(hp.NOTEBOOK_DIR / f'HOD_parameter_file_z{redshift:.3f}.yaml')

print(f'Halo catalog: {HALO_CATALOG}')
print(f'HODDIES catalog: {HODDIES_CATALOG}')
print(f'HMF output: {HMF_OUTPUT}')
print(f'Analysis directory: {ANALYSIS_DIR}')


In [ ]:
if not HODDIES_CATALOG.exists():
    raise FileNotFoundError(
        f'Missing {HODDIES_CATALOG}. Run jobs/halo_finder.sh on this run first, '
        'then come back to this notebook.'
    )

arr = np.load(HODDIES_CATALOG)
print(f'Loaded {len(arr):,} HODDIES-ready halos')
print(arr.dtype)

if HALO_METADATA.exists():
    with HALO_METADATA.open() as f:
        halo_metadata = json.load(f)
    print(json.dumps(halo_metadata, indent=2))
else:
    halo_metadata = {}

require_peculiar_velocities(halo_metadata, HALO_CATALOG)
boxsize = halo_metadata['boxsize']  # box size of the run, recorded by the halo finder
print(f'Box size: {boxsize} Mpc/h')

expected_cosmology = SERIALIZABLE_PARAMETERS
halo_cosmology = halo_metadata.get('cosmology')
if halo_cosmology is not None:
    mismatches = {
        key: (halo_cosmology.get(key), expected)
        for key, expected in expected_cosmology.items()
        if key in halo_cosmology and not np.isclose(float(halo_cosmology[key]), float(expected))
    }
    if mismatches:
        raise ValueError(f'Halo catalog cosmology metadata does not match Euclid reference: {mismatches}')
else:
    print('Halo catalog metadata has no cosmology block; using notebook Euclid reference cosmology.')


In [ ]:
print('Mass range:', arr['Mh'].min(), arr['Mh'].max())
print('log10_Mh range:', arr['log10_Mh'].min(), arr['log10_Mh'].max())
print('Position mins:', np.min(np.vstack([arr['x'], arr['y'], arr['z']]).T, axis=0))
print('Position maxs:', np.max(np.vstack([arr['x'], arr['y'], arr['z']]).T, axis=0))


In [ ]:
base_catalog = Base_catalogue({name: arr[name] for name in arr.dtype.names}, boxsize=boxsize, z_simu=redshift)
HOD_obj = HOD(base_catalog, param_file=PARAM_FILE)
HOD_obj.cosmo = euclid_cosmo  # used for the RSD of get_wp / get_xiells


In [ ]:
HOD_obj.plot_initial_HMF()

In [ ]:
cat = HOD_obj.make_mock_cat(tracers='LRG', fix_seed=42)

In [ ]:
HOD_obj.plot_HMF(cat,tracer='LRG')

In [ ]:
HOD_obj.plot_HMF(cat, show_sat=True, tracer='LRG')

In [ ]:
HOD_obj.HOD_plot()

In [ ]:
rp, wp = HOD_obj.get_wp(cat, tracers='LRG')
s, xi = HOD_obj.get_xiells(cat, tracers='LRG')

In [ ]:
HOD_obj.args['LRG']['link_sat_to_central'] = True

In [ ]:
plt.plot(rp,rp*wp)
plt.xscale('log')
plt.xlabel('$r_p$ [$\mathrm{{Mpc}}/h$]')
plt.ylabel(r'$r_p \cdot w_p(r_p)$ [$\mathrm{{Mpc}}/h$]$^2$')
plt.show()

In [ ]:
plt.plot(s,s*xi[0], label='l=0')
plt.plot(s,s*xi[1], label='l=2')
plt.xscale('log')
plt.xlabel('$s$ [$\mathrm{{Mpc}}/h$]')
plt.ylabel(r'$s \cdot \xi_{\ell}(s)$ [$\mathrm{{Mpc}}/h$]')
plt.legend()
plt.show()